In [9]:
import os, torch, random
import torchvision
import torchvision.transforms as transforms
from torch import nn
from torch.utils.data import DataLoader
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

BASE_DIR = r"D:\pycharm\CNN"
DATA_DIR = os.path.join(BASE_DIR, "data", "food-11")
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [10]:
tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                     [0.229, 0.224, 0.225])
])

train_ds = torchvision.datasets.ImageFolder(os.path.join(DATA_DIR, "training", "labeled"), transform=tf)
val_ds = torchvision.datasets.ImageFolder(os.path.join(DATA_DIR, "validation"), transform=tf)


unlabeled_ds = torchvision.datasets.ImageFolder(os.path.join(DATA_DIR, "training", "unlabeled"), transform=tf)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
unlabeled_loader = DataLoader(unlabeled_ds, batch_size=64, shuffle=False)

In [11]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256), nn.ReLU(),
            nn.Linear(256, 11)
        )

    def forward(self, x): return self.net(x)


model = Net().to(device)

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)


def evaluate():
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100 * correct / total



for epoch in range(10):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = criterion(model(x), y)
        optimizer.zero_grad();
        loss.backward();
        optimizer.step()


pseudo_imgs = []
pseudo_labels = []

model.eval()
with torch.no_grad():
    for x,_ in unlabeled_loader:
        x = x.to(device)
        prob = torch.softmax(model(x), 1)
        conf, pred = prob.max(1)

        mask = conf > 0.9
        if mask.sum() > 0:
            pseudo_imgs.append(x[mask].cpu())
            pseudo_labels.append(pred[mask].cpu())


if len(pseudo_imgs) > 0:
    px = torch.cat(pseudo_imgs)
    py = torch.cat(pseudo_labels)


    real_imgs = []
    real_labels = []

    for img, label in train_ds:
        real_imgs.append(img)
        real_labels.append(label)

    real_imgs = torch.stack(real_imgs)
    real_labels = torch.tensor(real_labels, dtype=torch.long)


    all_imgs = torch.cat([real_imgs, px])
    all_labels = torch.cat([real_labels, py])

    new_dataset = torch.utils.data.TensorDataset(all_imgs, all_labels)
    train_loader = DataLoader(new_dataset, batch_size=64, shuffle=True)

    print("Pseudo-label added:", len(px))
else:
    print("No pseudo-label generated, skip")


best = 0

for epoch in range(10):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    acc = evaluate()
    print(f"Epoch {epoch+1}: {acc:.2f}%")

    if acc > best:
        best = acc
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, "level3.pth"))

print("Best:", best)

Pseudo-label added: 1837
Epoch 1: 33.79%
Epoch 2: 35.45%
Epoch 3: 35.76%
Epoch 4: 36.06%
Epoch 5: 35.91%
Epoch 6: 35.30%
Epoch 7: 35.15%
Epoch 8: 35.61%
Epoch 9: 35.45%
Epoch 10: 35.61%
Best: 36.06060606060606
